In [ ]:
import pandas as pd
import numpy as np

pd.set_option('max_colwidth', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

plt.rcParams['figure.figsize'] = (12, 8)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from ipywidgets import interact

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [ ]:
data=pd.read_csv('Crop_recommendation.csv')
data.head()

In [ ]:
plt.figure(figsize=(15,10))

plt.subplot(2,4,1)
sns.histplot(data['N'], kde=True, color='orange')
plt.title("Ratio of Nitrogen")

plt.subplot(2,4,2)
sns.histplot(data['P'], kde=True, color='blue')
plt.title("Ratio of Phosphorous")

plt.subplot(2,4,3)
sns.histplot(data['K'], kde=True, color='pink')
plt.title("Ratio of Potassium")

plt.subplot(2,4,4)
sns.histplot(data['temperature'], kde=True, color='green')
plt.title("Ratio of Temperature")
plt.subplot(2,4,5)
sns.histplot(data['humidity'], kde=True, color='blue')
plt.title("Ratio of Humidity")

plt.subplot(2,4,6)
sns.histplot(data['ph'], kde=True, color='red')
plt.title("Ratio of PH")

plt.subplot(2,4,7)
sns.histplot(data['rainfall'], kde=True, color='yellow')
plt.title("Ratio of Rainfall")

plt.suptitle("Distribution of agricultural conditions", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
columns = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
colors = ['orange', 'blue', 'pink', 'green', 'blue', 'red', 'yellow']
titles = [
    'Ratio of Nitrogen',
    'Ratio of Phosphorous',
    'Ratio of Potassium',
    'Ratio of Temperature',
    'Ratio of Humidity',
    'Ratio of PH',
    'Ratio of Rainfall'
]
plt.figure(figsize=(15,10))

for i, col in enumerate(columns):
    plt.subplot(2,4,i+1)
    sns.histplot(data[col], kde=True, color=colors[i])
    plt.title(titles[i])

plt.suptitle("Distribution of agricultural conditions", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,8))

plt.subplot(2,4,7)
sns.scatterplot(x=data['humidity'], y=data['label'])

plt.show()

In [ ]:
data.columns

In [ ]:
data = data.rename(columns={
    'N':'nitrogen',
    'P':'phosphorous',
    'K':'potassium'
})

In [ ]:
plot_df = data[['nitrogen','phosphorous','potassium','temperature','humidity','ph','rainfall']]

plot_df = plot_df.melt(var_name='Feature', value_name='Value')

plt.figure(figsize=(12,6))
sns.countplot(
    x='Feature',
    data=plot_df,
    hue='Feature',
    palette='deep',
    legend=False
)

plt.show()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(data=data[['nitrogen','phosphorous','potassium','temperature','humidity','ph','rainfall']])

plt.xticks(
    range(7),
    ['nitrogen','phosphorous','potassium','temperature','humidity','ph','rainfall']
)

plt.show()

In [ ]:
Q1 = data['phosphorous'].quantile(0.25)
Q3 = data['phosphorous'].quantile(0.75)
IQR = Q3 - Q1

filter = (data['phosphorous'] >= Q1 - 1.5*IQR) & (data['phosphorous'] <= Q3 + 1.5*IQR)

data = data.loc[filter]


In [ ]:
Q1 = data['potassium'].quantile(0.25)
Q3 = data['potassium'].quantile(0.75)
IQR = Q3 - Q1

filter = (data['potassium'] >= Q1 - 1.5*IQR) & (data['potassium'] <= Q3 + 1.5*IQR)

data = data.loc[filter]

In [ ]:
Q1 = data['rainfall'].quantile(0.25)
Q3 = data['rainfall'].quantile(0.75)
IQR = Q3 - Q1

filter = (data['rainfall'] >= Q1 - 1.5*IQR) & (data['rainfall'] <= Q3 + 1.5*IQR)

data=data.loc[filter]

In [ ]:
print("Summer crops")
print(data[(data['temperature'] > 30) & (data['humidity'] > 50)]['label'].unique())
print("----------------------------------------------")

print("Winter crops")
print(data[(data['temperature'] < 20) & (data['humidity'] > 30)]['label'].unique())
print("----------------------------------------------")

print("Rainy crops")
print(data[(data['rainfall'] > 200) & (data['humidity'] > 50)]['label'].unique())
print("----------------------------------------------")

In [ ]:
y = data['label']
x = data.drop(['label'], axis=1)

print("Shape of x:", x.shape)
print("Shape of y:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=0
)

print("The shape of x_train:", x_train.shape)
print("The shape of x_test:", x_test.shape)
print("The shape of y_train:", y_train.shape)
print("The shape of y_test:", y_test.shape)

In [ ]:
# Elbow method used to find out the optimum number of clusters

plt.rcParams['figure.figsize'] = (10,4)

wcss = []

for i in range(1,11):
    km = KMeans(n_clusters=i,
                init='k-means++',
                max_iter=300,
                n_init=10,
                random_state=0)
    km.fit(x)
    wcss.append(km.inertia_)

plt.plot(range(1,11), wcss)
plt.title("The Elbow Method", fontsize=20)
plt.xlabel("No of Clusters")
plt.ylabel("WCSS")
plt.show()

In [ ]:
km = KMeans(n_clusters=4,
            init='k-means++',
            max_iter=300,
            n_init=10,
            random_state=0)

y_means = km.fit_predict(x)

a = data['label']

y_means = pd.DataFrame(y_means)

z = pd.concat([y_means, a], axis=1)

z = z.rename(columns={0:'cluster'})
print("lets check the results after applying the K-Means clustering analysis \n")

print("Crops in First Cluster:", z[z['cluster']==0]['label'].unique())

print("------------------------------------------------------------")

print("Crops in Second Cluster:", z[z['cluster']==1]['label'].unique())

print("------------------------------------------------------------")

print("Crops in Third Cluster:", z[z['cluster']==2]['label'].unique())
print("------------------------------------------------------------")

print("Crops in Fourth Cluster:", z[z['cluster']==3]['label'].unique())

In [ ]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression()
model.fit(x_train,y_train)
y_pred=model.predict(x_test)

In [ ]:
from sklearn.metrics import classification_report
cr=classification_report(y_pred,y_test)
print(cr)

In [ ]:
prediction=model.predict((np.array([[105,35,40,25,64,7,160]])))
print("The suggested crop for given climate condition is:",prediction)

In [ ]:
import os

os.mkdir("CropPrediction")